# Exercise 4 — Train a FastConformer-CTC (Colab GPU)

A runnable walkthrough of the lab on a free Colab T4. The cells clone the repo and run the **reference scripts** step by step. For a full-quality run (≤10% WER) you need `train-clean-100` and a longer schedule — see the README and `../CLOUD_GPU_SETUP.md`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cwkendall/parakeet-study/blob/main/exercises/04-train-fastconformer-ctc/explore.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/cwkendall/parakeet-study/main?labpath=exercises%2F04-train-fastconformer-ctc%2Fexplore.ipynb)

### Step 0 — confirm a GPU is attached

In Colab: **Runtime → Change runtime type → GPU**.

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
!nvidia-smi -L

### Step 1 — clone the repo and enter the lab

The scripts you run below are the full reference solution.

In [ ]:
![ -d parakeet-study ] || git clone https://github.com/cwkendall/parakeet-study.git
%cd parakeet-study/exercises/04-train-fastconformer-ctc

### Step 2 — install dependencies (a few minutes)

In [ ]:
!pip -q install -r requirements.txt

### Step 3 — download a small slice of LibriSpeech

Full `train-clean-100` is large; start with `dev-clean` + `test-clean` to validate the pipeline, then add `train-clean-100` for a real run.

In [ ]:
!python download_librispeech.py --data_root ./data --data_sets dev-clean,test-clean

### Step 4 — train a BPE tokenizer

In [ ]:
!python train_tokenizer.py \
  --manifest ./data/dev-clean.json \
  --vocab_size 128 --out_dir ./tokenizer

### Step 5 — smoke-test training

A few steps to confirm the config + data wire up. For the real run, raise `trainer.max_steps` and add `train-clean-100` (see the README and `../CLOUD_GPU_SETUP.md`).

In [ ]:
!python train.py \
  --config-path ./conf --config-name fastconformer_ctc_small \
  trainer.max_steps=50 trainer.val_check_interval=50

### Step 6 — evaluate WER

In [ ]:
!python evaluate.py \
  --model ./nemo_experiments/**/checkpoints/*.nemo \
  --manifest ./data/test-clean.json || echo 'train longer first for a meaningful WER'

---

When the smoke test passes, scale up the data and `max_steps` for a real run. The READMEs have the target metrics and reflection questions.